![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-4-h-micro` to analyze eXtensive Business Reporting Language (XBRL) tags of financial reports

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of tag entity extraction in watsonx. It introduces commands for data retrieval, model testing and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to use `ibm/granite-4-h-micro` model to analyze XBRL tags of financial phrases.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on IBM watsonx.ai](#Foundation-Models-on-IBM-watsonx.ai)
4. [Analyze the report](#Analyze-the-report)
5. [Score the model](#Score-the-model)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U datasets | tail -n 1
%pip install -U "scikit-learn==1.6.1" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

### Working with projects

First of all, you need to create a project that will be used for your work. If you do not have a project created already, follow the steps below:

- Open IBM Cloud Pak® main page
- Click all projects
- Create an empty project
- Copy `project_id` from url and paste it below

**Action**: Assign project ID below

In [4]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id)

<a id="Data-loading"></a>
## Data loading

Download the `nlpaueb/finer-139` dataset

In [6]:
from datasets import load_dataset

finer_train = load_dataset("nlpaueb/finer-139", split="train")

Retrieve the entity tags

In [7]:
ner_tags = finer_train.features["ner_tags"].feature.names
ner_tags[:10]

['O',
 'B-AccrualForEnvironmentalLossContingencies',
 'B-AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife',
 'I-AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife',
 'B-AllocatedShareBasedCompensationExpense',
 'B-AmortizationOfFinancingCosts',
 'B-AmortizationOfIntangibleAssets',
 'I-AmortizationOfIntangibleAssets',
 'B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount',
 'I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount']

Gather data, which references a single tag

In [8]:
single_tag_indexes: dict[int, int] = {}

for i, tags in enumerate(finer_train["ner_tags"]):
    tag_count = sum(tag != 0 for tag in tags)
    if tag_count == 1:
        single_tag_indexes[i] = next(item for item in tags if item)

list(single_tag_indexes.items())[:20]

[(26, 168),
 (34, 14),
 (41, 41),
 (43, 41),
 (45, 41),
 (72, 87),
 (73, 87),
 (74, 87),
 (84, 41),
 (86, 41),
 (89, 41),
 (90, 41),
 (92, 41),
 (93, 41),
 (125, 93),
 (126, 93),
 (127, 72),
 (170, 67),
 (194, 67),
 (201, 85)]

Convert the tokens into sequences for the model

In [9]:
sequences = [
    " ".join(token)
    for i, token in enumerate(finer_train["tokens"])
    if i in single_tag_indexes
]

Inspect exemplary sequence

In [10]:
sequences[3]

'Interest on the 7.00 % Senior Notes is due semi - annually .'

Select test sequences

In [11]:
from collections import defaultdict

sequences_for_tag: defaultdict[str, list[str]] = defaultdict(list)
for sequence, tag_index in zip(sequences, single_tag_indexes.values()):
    tag_value = ner_tags[tag_index]

    sequences_for_tag[tag_value].append(sequence)

selected_ner_tags = list(sequences_for_tag)[:10]

<a id="Foundation-Models-on-IBM-watsonx.ai"></a>
## Foundation Models on IBM watsonx.ai

#### List available models

In [12]:
for model in client.foundation_models.ChatModels:
    print(f"- {model}")

- ibm/granite-4-h-micro
- ibm/ibm-defense-3-3-8b-instruct
- magistral-small-2509
- meta-llama/llama-3-2-1b-instruct
- ministral-8b-instruct-2512
- mistralai/mistral-small-3-2-24b-instruct-2506


You need to specify `model_id` that will be used for inferencing:

In [13]:
model_id = client.foundation_models.ChatModels.GRANITE_4_H_MICRO

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [14]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames as GenParams

parameters = {
    GenParams.TEMPERATURE: 0,
    GenParams.REPETITION_PENALTY: 1,
}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [15]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(model_id=model_id, params=parameters, api_client=client)

### Model's details

In [16]:
model.get_details()

{'model_id': 'ibm/granite-4-h-micro',
 'label': 'granite-4-h-micro',
 'provider': 'IBM',
 'source': 'IBM',
 'functions': [{'id': 'text_chat'}],
 'short_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets.',
 'long_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets. This model is developed using a diverse set of techniques with a structured chat format, including supervised finetuning, model alignment using reinforcement learning, and model merging. Granite 4.0 instruct models feature improved instruction following (IF) and tool-calling capabilities, making them more effective in enterprise applications.'

<a id="Analyze-the-report"></a>
## Analyze the report

Define instructions for the model. 

**Hint:** All possible tags must be attached in the instruction

In [17]:
def get_messages(
    example_reports: list[str],
    example_tags: list[str],
    all_tags: list[str],
    report_to_check: str,
) -> list[dict[str, str]]:
    system_content = (
        "You are a tag classification model. "
        "Determine the eXtensive Business Reporting Language tag in the financial report from following tags. "
        "The prefix `B` means it is a balance sheet item, while prefix `I` means it is an income statement item. "
        "The following tags are available, produce only a single tag:\n"
    )
    system_content += "\n".join(f"- {tag}" for tag in all_tags)

    messages = [{"role": "system", "content": system_content}]

    for report, tag in zip(example_reports, example_tags):
        messages.append({"role": "user", "content": report})
        messages.append({"role": "assistant", "content": tag})

    messages.append({"role": "user", "content": report_to_check})

    return messages

Prepare few-shot examples.

In [18]:
example_tags: list[str] = sorted(selected_ner_tags)
example_reports: list[str] = [sequences_for_tag[tag][0] for tag in example_tags]

print(example_tags)
print(*example_reports, sep="\n")

['B-AreaOfRealEstateProperty', 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired', 'B-DebtInstrumentInterestRateStatedPercentage', 'B-FiniteLivedIntangibleAssetUsefulLife', 'B-GuaranteeObligationsMaximumExposure', 'B-LineOfCreditFacilityCurrentBorrowingCapacity', 'B-LineOfCreditFacilityMaximumBorrowingCapacity', 'B-LossContingencyDamagesSoughtValue', 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross', 'B-UnrecognizedTaxBenefits']
The East Rutherford facility consisted of warehouses and offices totaling approximately 81,000 square feet of space .
In 2010 , the Rialto segment acquired indirectly 40 % managing member equity interests in two limited liability companies ( " LLCs " ) in partnership with the FDIC ( “ FDIC Portfolios ” ) .
Rialto used the net proceeds of the 7.00 % Senior Notes to provide additional working capital for RMF , and to make investments in the funds that Rialto manages , as well as for general corporate purposes .
Patent

### Analyze financial phrase eXtensive Business Reporting Language using `ibm/granite-4-h-micro` model.

Analyze the report

In [19]:
results: list[str] = []
for tag in selected_ner_tags:
    messages = get_messages(
        example_reports,
        example_tags,
        selected_ner_tags,
        sequences_for_tag[tag][1],
    )

    response = model.chat(messages)
    results.append(response["choices"][0]["message"]["content"])

results

['B-UnrecognizedTaxBenefits',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-DebtInstrumentInterestRateStatedPercentage',
 'B-DebtInstrumentInterestRateStatedPercentage',
 'B-LossContingencyDamagesSoughtValue',
 'B-GuaranteeObligationsMaximumExposure',
 'B-FiniteLivedIntangibleAssetUsefulLife',
 'B-LineOfCreditFacilityCurrentBorrowingCapacity',
 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross',
 'B-AreaOfRealEstateProperty']

<a id="Score-the-model"></a>
## Score the model

**Note:** To run the Score section for model scoring please transform following `markdown` cells to `code` cells.
Have in mind that the score is calculated only on dataset sample, for relevant performance metric please score the model on the whole `nlpaueb/finer-139` dataset.

Get the true labels.

In [20]:
y_true = selected_ner_tags
y_true

['B-UnrecognizedTaxBenefits',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-DebtInstrumentInterestRateStatedPercentage',
 'B-LineOfCreditFacilityMaximumBorrowingCapacity',
 'B-LossContingencyDamagesSoughtValue',
 'B-GuaranteeObligationsMaximumExposure',
 'B-FiniteLivedIntangibleAssetUsefulLife',
 'B-LineOfCreditFacilityCurrentBorrowingCapacity',
 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross',
 'B-AreaOfRealEstateProperty']

Get the sentiment labels returned by the `ibm/granite-guardian-3-8b` model.

In [21]:
y_pred = [result.strip() for result in results]
y_pred

['B-UnrecognizedTaxBenefits',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-DebtInstrumentInterestRateStatedPercentage',
 'B-DebtInstrumentInterestRateStatedPercentage',
 'B-LossContingencyDamagesSoughtValue',
 'B-GuaranteeObligationsMaximumExposure',
 'B-FiniteLivedIntangibleAssetUsefulLife',
 'B-LineOfCreditFacilityCurrentBorrowingCapacity',
 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross',
 'B-AreaOfRealEstateProperty']

Calculate accuracy score.

In [22]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_pred, y_true))

0.9


<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to predict the financial phrases XBRL tag with `ibm/granite-4-h-micro` on watsonx. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.